In [2]:
import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

CUDA: True
GPU: Tesla T4
CUDA version: 13.0


In [3]:
!pip install -q uv
!git clone https://github.com/jaredpalmer/kev.git
%cd kev
!uv sync --extra serve

Cloning into 'kev'...
remote: Enumerating objects: 11687, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 11687 (delta 125), reused 70 (delta 70), pack-reused 11530 (from 2)
Receiving objects: 100% (11687/11687), 178.48 MiB | 27.50 MiB/s, done.
Resolving deltas: 100% (7148/7148), done.
Updating files: 100% (3057/3057), done.
/kaggle/working/kev
Using CPython 3.13.16 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 120 packages in 1ms
Prepared 113 packages in 5m 19s                                          
░░░░░░░░░░░░░░░░░░░░ [0/113] Installing wheels...                               warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to supp

In [4]:
%cd /kaggle/working/kev

!uv pip install --python .venv/bin/python wrapt

/kaggle/working/kev
Resolved 1 package in 1.67s                                          
Prepared 1 package in 73ms                                               
░░░░░░░░░░░░░░░░░░░░ [0/1] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 4ms                                  
 + wrapt==2.5.2


In [5]:
!uv run python -m kev.train --help

usage: train.py [-h] [--base BASE] [--n_per_source N_PER_SOURCE]
                [--epochs EPOCHS] [--lr LR] [--head_lr HEAD_LR]
                [--weight_decay WEIGHT_DECAY] [--lora LORA] [--accum ACCUM]
                [--holdout HOLDOUT] [--perm_kl PERM_KL]
                [--perm_frac PERM_FRAC] [--ord_w ORD_W]
                [--label_smoothing LABEL_SMOOTHING] [--brier_w BRIER_W]
                [--focal_gamma FOCAL_GAMMA] [--suite SUITE]
                [--train_sources TRAIN_SOURCES] [--device {auto,cpu,cuda,mps}]
                [--batch BATCH] [--dtype {fp32,bf16}]
                [--weights_dtype {fp32,bf16}] [--checkpointing {0,1}]
                [--option_isolation {0,1}] [--special_embeddings {0,1}]
                [--head_dim HEAD_DIM] [--lora_targets {all,dense,attn,qv}]
                [--base_revision BASE_REVISION] [--p_none P_NONE]
                [--p_none_distract P_NONE_DISTRACT] [--p_distract P_DISTRACT]
                [--p_none_pair P_NONE_PAIR]
             

In [6]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("*.jsonl"):
    print(p)

/kaggle/input/datasets/nihaoyongshichen/local-mind/train.jsonl
/kaggle/input/datasets/nihaoyongshichen/local-mind/validation.jsonl


In [ ]:
import subprocess
import time
import requests

log = open("/kaggle/working/kev_server.log", "w")

server = subprocess.Popen(
    [
        "uv", "run", "--extra", "serve",
        "python", "-m", "kev.serve",
        "--run", "jaredpalmer/kev-0.8b",
        "--device", "cuda",
        "--port", "8009",
    ],
    stdout=log,
    stderr=subprocess.STDOUT
)

url = "http://127.0.0.1:8009"

for i in range(180):
    if server.poll() is not None:
        print("Kev 服务启动失败，请查看日志")
        break
    try:
        response = requests.get(
            url + "/openapi.json",
            timeout=2
        )
        if response.ok:
            print("Kev server connected!")
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print("服务尚未就绪，请查看日志")

Kev server connected!


In [ ]:
import requests

payload = {
    "model": "local",
    "state": (
        "The brake pedal feels soft and the vehicle "
        "takes much longer than usual to stop."
    ),
    "questions": {
        "brakes": {
            "type": "noul",
            "instructions": (
                "Does this consumer vehicle safety complaint "
                "concern the service brakes or braking system?"
            )
        }
    }
}

response = requests.post(
    "http://127.0.0.1:8009/v1/systemone",
    json=payload,
    timeout=120
)

response.raise_for_status()
print(response.json())

{'model': 'local', 'answers': {'brakes': {'type': 'noul', 'noul': 0.8044}}, 'usage': {'input_tokens': 39, 'output_tokens': 23}, 'latency_ms': 265.6}


In [ ]:
import json
from pathlib import Path
import requests
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
)

validation_files = list(
    Path("/kaggle/input").rglob("validation.jsonl")
)

assert len(validation_files) == 1, (
    f"需要找到唯一的 validation.jsonl，实际找到: {validation_files}"
)

validation_path = validation_files[0]
print("Validation:", validation_path)

labels = []
probabilities = []

with validation_path.open(encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 100:
            break

        example = json.loads(line)
        question = example["questions"]["brakes"]

        payload = {
            "model": "local",
            "state": example["state"],
            "questions": {
                "brakes": {
                    "type": "noul",
                    "instructions": question["instructions"]
                }
            }
        }

        result = requests.post(
            "http://127.0.0.1:8009/v1/systemone",
            json=payload,
            timeout=120
        )
        result.raise_for_status()

        probability = result.json()["answers"]["brakes"]["noul"]

        labels.append(int(question["label"]))
        probabilities.append(float(probability))

predictions = [int(p >= 0.5) for p in probabilities]

print("Samples:", len(labels))
print("Positive labels:", sum(labels))
print("Accuracy:", accuracy_score(labels, predictions))
print("Brier:", brier_score_loss(labels, probabilities))

if len(set(labels)) == 2:
    print("PR-AUC:", average_precision_score(labels, probabilities))
else:
    print("PR-AUC: 当前样本只有一个类别，暂不计算")

Validation: /kaggle/input/datasets/nihaoyongshichen/local-mind/validation.jsonl
Samples: 100
Positive labels: 12
Accuracy: 0.85
Brier: 0.13475540409999998
PR-AUC: 0.7265952605451117


In [ ]:
server.terminate()
server.wait(timeout=30)
print("Original Kev server stopped.")

Original Kev server stopped.


In [ ]:
import json
from pathlib import Path

train_files = list(Path("/kaggle/input").rglob("train.jsonl"))
assert len(train_files) == 1, train_files

output = Path("/kaggle/working/train_1000.jsonl")

with train_files[0].open(encoding="utf-8") as source, \
     output.open("w", encoding="utf-8") as target:
    for i, line in enumerate(source):
        if i >= 1000:
            break
        json.loads(line)  # 检查 JSON 格式
        target.write(line)

print("Training sample ready:", output)

Training sample ready: /kaggle/working/train_1000.jsonl


In [ ]:
%cd /kaggle/working/kev
!uv run python -m kev.train --help

/kaggle/working/kev
usage: train.py [-h] [--base BASE] [--n_per_source N_PER_SOURCE]
                [--epochs EPOCHS] [--lr LR] [--head_lr HEAD_LR]
                [--weight_decay WEIGHT_DECAY] [--lora LORA] [--accum ACCUM]
                [--holdout HOLDOUT] [--perm_kl PERM_KL]
                [--perm_frac PERM_FRAC] [--ord_w ORD_W]
                [--label_smoothing LABEL_SMOOTHING] [--brier_w BRIER_W]
                [--focal_gamma FOCAL_GAMMA] [--suite SUITE]
                [--train_sources TRAIN_SOURCES] [--device {auto,cpu,cuda,mps}]
                [--batch BATCH] [--dtype {fp32,bf16}]
                [--weights_dtype {fp32,bf16}] [--checkpointing {0,1}]
                [--option_isolation {0,1}] [--special_embeddings {0,1}]
                [--head_dim HEAD_DIM] [--lora_targets {all,dense,attn,qv}]
                [--base_revision BASE_REVISION] [--p_none P_NONE]
                [--p_none_distract P_NONE_DISTRACT] [--p_distract P_DISTRACT]
                [--p_none_pair P_NONE

In [ ]:
!uv run python -m kev.train \
  --data /kaggle/working/train_1000.jsonl \
  --base Qwen/Qwen3.5-0.8B-Base \
  --init_from jaredpalmer/kev-0.8b \
  --epochs 1 \
  --lr 2e-5 \
  --batch 1 \
  --accum 8 \
  --dtype fp32 \
  --checkpointing 1 \
  --device cuda \
  --out /kaggle/working/localmind-test

Loading weights: 100%|██████████████████████| 320/320 [00:00<00:00, 368.54it/s]
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 1234.83it/s]
Download complete:                                         |  0.00B            
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            
delta: warm start from /root/.cache/huggingface/hub/models--jaredpalmer--kev-0.8b/snapshots/bf75a6a8848ea6960ff2ed108d9ed44c2941174f: 372 lora tensors and the pointer head loaded
device=cuda world=1 trainable params=11.3M
dropped 49 of 1000 records that exceed the training context (384 state / 1024 branch / 2048 packed tokens)
951 training requests (holdout=[]), questions by type {'noul': 951}
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not i

In [ ]:
import json
from pathlib import Path

folder = Path("/kaggle/working/localmind-test")

for filename in ["training_config.json", "training_metrics.json"]:
    p = folder / filename
    print(f"\n===== {filename} =====")

    if p.exists():
        print(json.dumps(
            json.loads(p.read_text()),
            indent=2,
            ensure_ascii=False
        ))


===== training_config.json =====
{
  "args": {
    "base": "Qwen/Qwen3.5-0.8B-Base",
    "n_per_source": 1000,
    "epochs": 1,
    "lr": 2e-05,
    "head_lr": 0.0,
    "weight_decay": 0.01,
    "lora": 16,
    "accum": 8,
    "holdout": "",
    "perm_kl": 0.0,
    "perm_frac": 0.3,
    "ord_w": 0.0,
    "label_smoothing": 0.0,
    "brier_w": 0.0,
    "focal_gamma": 0.0,
    "suite": null,
    "train_sources": "",
    "device": "cuda",
    "batch": 1,
    "dtype": "fp32",
    "weights_dtype": "fp32",
    "checkpointing": 1,
    "option_isolation": 0,
    "special_embeddings": 0,
    "head_dim": 256,
    "lora_targets": "all",
    "base_revision": "",
    "p_none": 0.1,
    "p_none_distract": 0.12,
    "p_distract": 0.15,
    "p_none_pair": 0.0,
    "none_pair_max_state": null,
    "synthetic_repeat": 1,
    "public_frac": 1.0,
    "anchor": "",
    "anchor_w": 0.0,
    "anchor_sources": "",
    "out": "/kaggle/working/localmind-test",
    "data": "/kaggle/working/train_1000.jsonl",
  

In [ ]:
import subprocess
import time
import requests

log = open("/kaggle/working/localmind_server.log", "w")

localmind_server = subprocess.Popen(
    [
        "uv", "run", "--extra", "serve",
        "python", "-m", "kev.serve",
        "--run", "/kaggle/working/localmind-test",
        "--device", "cuda",
        "--port", "8010"
    ],
    cwd="/kaggle/working/kev",
    stdout=log,
    stderr=subprocess.STDOUT
)

for _ in range(90):
    if localmind_server.poll() is not None:
        print("模型加载失败，请检查日志。")
        break

    try:
        r = requests.get("http://127.0.0.1:8010/openapi.json", timeout=2)
        if r.ok:
            print("LocalMind-Kev connected!")
            break
    except requests.RequestException:
        pass

    time.sleep(2)
else:
    print("模型尚未就绪。")

LocalMind-Kev connected!


In [16]:
import json
import requests
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score
)

validation_path = list(
    Path("/kaggle/input").rglob("validation.jsonl")
)[0]

labels = []
probabilities = []

with validation_path.open(encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 100:
            break

        example = json.loads(line)
        question = example["questions"]["brakes"]

        payload = {
            "model": "local",
            "state": example["state"],
            "questions": {
                "brakes": {
                    "type": "noul",
                    "instructions": question["instructions"]
                }
            }
        }

        result = requests.post(
            "http://127.0.0.1:8010/v1/systemone",
            json=payload,
            timeout=120
        )
        result.raise_for_status()

        probability = float(
            result.json()["answers"]["brakes"]["noul"]
        )

        labels.append(int(question["label"]))
        probabilities.append(probability)

predictions = [int(p >= 0.5) for p in probabilities]

print("===== Fine-tuned LocalMind-Kev =====")
print("Samples:", len(labels))
print("Positive labels:", sum(labels))
print("Accuracy:", round(accuracy_score(labels, predictions), 4))
print("Brier:", round(brier_score_loss(labels, probabilities), 4))
print("PR-AUC:", round(average_precision_score(labels, probabilities), 4))
print("Precision:", round(precision_score(labels, predictions, zero_division=0), 4))
print("Recall:", round(recall_score(labels, predictions, zero_division=0), 4))
print("F1:", round(f1_score(labels, predictions, zero_division=0), 4))

===== Fine-tuned LocalMind-Kev =====
Samples: 100
Positive labels: 12
Accuracy: 0.94
Brier: 0.0483
PR-AUC: 0.8686
Precision: 0.7143
Recall: 0.8333
F1: 0.7692


In [17]:
import shutil

shutil.make_archive(
    "/kaggle/working/localmind-kev-first",
    "zip",
    "/kaggle/working/localmind-test"
)

print("模型已打包：/kaggle/working/localmind-kev-first.zip")

模型已打包：/kaggle/working/localmind-kev-first.zip


In [18]:
import shutil

shutil.make_archive(
    "/kaggle/working/localmind-kev-first",
    "zip",
    "/kaggle/working/localmind-test"
)

print("Checkpoint backup created!")

Checkpoint backup created!
